# Two-Stage Doctor-Patient Matching System

This notebook implements a sophisticated two-stage recommendation system to match patients with doctors. This approach balances efficiency with high-precision ranking.

- **Stage 1: Candidate Generation:** Uses a lightweight model (Matrix Factorization) to quickly filter thousands of providers down to a smaller, relevant candidate pool.
- **Stage 2: Detailed Ranking:** Uses a powerful, feature-rich model (Random Forest) to precisely rank the candidates from Stage 1.

## Setup: Loading Data and Installing Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import TruncatedSVD
import subprocess
import sys

# Install necessary libraries for feature engineering
try:
    from ethnicolr import census_ln
except ImportError:
    print("Installing ethnicolr...")
    subprocess.run([sys.executable, "-m", "pip", "install", "ethnicolr"], check=True)
    from ethnicolr import census_ln

try:
    import pgeocode
except ImportError:
    print("Installing pgeocode...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pgeocode"], check=True)
    import pgeocode

# --- IMPORTANT: Update these file paths to match your system ---
parquet_file_paths = {
    "patient": "path/to/your/synthetic_patients.parquet",
    "encounter": "path/to/your/synthetic_encounters.parquet",
    "provider": "path/to/your/synthetic_providers.parquet"
}

# Reading the parquet files
patient_df = pd.read_parquet(parquet_file_paths['patient'])
encounter_df = pd.read_parquet(parquet_file_paths['encounter'])
provider_df = pd.read_parquet(parquet_file_paths['provider'])

print("Dataframes loaded successfully.")

## Common Feature Engineering
This step derives the provider's race, which is a crucial feature for the ranking model.

In [ ]:
# Derive race for the provider_df as it is missing from the source data
print("Deriving race for providers from last names...")
race_predictions = census_ln(provider_df, 'last_name')
race_cols = ['race_white', 'race_black', 'race_api', 'race_aian', 'race_2prace']
provider_df['derived_race'] = race_predictions[race_cols].idxmax(axis=1).str.replace('race_', '')
print("Provider race derivation complete.")

---

## Stage 1: Candidate Generation Model (Matrix Factorization)

In [ ]:
print("Training Stage 1 Model: Matrix Factorization...")

# Create a patient-provider interaction matrix using satisfaction scores
interaction_df = encounter_df.pivot_table(index='patient_id', columns='provider_id', values='patient_satisfaction')

# Fill missing values (where a patient hasn't seen a provider) with the average satisfaction
# This helps the model learn a baseline
mean_satisfaction = encounter_df['patient_satisfaction'].mean()
interaction_df_filled = interaction_df.fillna(mean_satisfaction)

# Create mappings from ID to matrix index
patient_id_map = {id: i for i, id in enumerate(interaction_df_filled.index)}
provider_id_map = {id: i for i, id in enumerate(interaction_df_filled.columns)}
# Create reverse mappings to get IDs back
patient_idx_map = {i: id for id, i in patient_id_map.items()}

# Use TruncatedSVD for Matrix Factorization
svd = TruncatedSVD(n_components=50, random_state=42) # n_components = number of latent features
svd.fit(interaction_df_filled.values)

# The trained SVD model is now ready to generate candidates.
print("Stage 1 Model (SVD) training complete.")

---

## Stage 2: Detailed Ranking Model (Learning-to-Rank)

In [ ]:
print("Training Stage 2 Model: Learning-to-Rank (Random Forest)...")

# --- 1. Prepare Data for LTR Model ---
# Merge all data into a single master DataFrame
master_df = pd.merge(encounter_df, patient_df, on='patient_id')
master_df = pd.merge(master_df, provider_df, on='provider_id')

# Engineer detailed features
master_df['language_match'] = [1 if p_lang in d_langs else 0 for p_lang, d_langs in zip(master_df['primary_language'], master_df['languages_spoken'])]
master_df['race_match'] = (master_df['race'] == master_df['derived_race']).astype(int)

# Add geographic distance (this can take a moment on the first run)
dist = pgeocode.GeoDistance('US')
zip_distances = dist.query_postal_code(master_df['zip_code'].tolist(), master_df['practice_zip_code'].tolist())
master_df['distance_km'] = zip_distances
master_df['distance_km'].fillna(master_df['distance_km'].mean(), inplace=True) # Fill any missing distances

# --- 2. Train the LTR Model ---
features_ltr = ['cultural_competency_rating', 'years_experience', 'language_match', 'race_match', 'distance_km']
target_ltr = 'patient_satisfaction'

X = master_df[features_ltr]
y = master_df[target_ltr]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ltr_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
ltr_model.fit(X_train, y_train)

# --- 3. Evaluate the LTR Model ---
predictions = ltr_model.predict(X_test)
rmse = mean_squared_error(y_test, predictions, squared=False)
print(f"Stage 2 Model (Random Forest) Validation RMSE: {rmse:.4f}")

print("\nLearned Feature Importances for Ranking Model:")
importances = pd.Series(ltr_model.feature_importances_, index=features_ltr).sort_values(ascending=False)
print(importances)

---

## Putting It All Together: The Two-Stage Recommendation Function

In [ ]:
def get_two_stage_recommendations(patient_id, required_specialty, 
                                all_providers_df, all_patients_df, 
                                svd_model, ltr_model, 
                                interaction_matrix, patient_map, provider_map, 
                                num_candidates=100, num_final_recs=5):
    """
    Generates recommendations using the two-stage filtering and ranking process.
    """
    print(f"\n--- Generating recommendations for Patient ID: {patient_id} ---")
    patient_info = all_patients_df[all_patients_df['patient_id'] == patient_id].iloc[0]
    
    # --- STAGE 1: CANDIDATE GENERATION ---
    print(f"Stage 1: Finding top {num_candidates} candidates...")
    # 1a. Apply hard filters
    candidate_providers = all_providers_df[all_providers_df['specialty'] == required_specialty].copy()
    
    # 1b. Use SVD model for initial scoring
    patient_idx = patient_map.get(patient_id)
    if patient_idx is None:
        # Cold start: if patient is new, we can't use SVD. Skip to stage 2.
        print("New patient (cold start). Skipping Stage 1 SVD filtering.")
        stage1_candidates = candidate_providers
    else:
        patient_latent_vector = svd_model.transform(interaction_matrix.iloc[patient_idx, :].values.reshape(1, -1))
        provider_latent_vectors = svd_model.components_.T
        # Predict scores for all providers
        svd_scores = np.dot(patient_latent_vector, provider_latent_vectors.T).flatten()
        
        # Add scores to providers that passed hard filters
        candidate_providers['svd_score'] = candidate_providers['provider_id'].map(lambda x: svd_scores[provider_map.get(x, -1)])
        candidate_providers.dropna(subset=['svd_score'], inplace=True)
        
        # Get top K candidates
        stage1_candidates = candidate_providers.nlargest(num_candidates, 'svd_score')

    # --- STAGE 2: DETAILED RANKING ---
    print(f"Stage 2: Ranking {len(stage1_candidates)} candidates with LTR model...")
    if stage1_candidates.empty:
        return "No suitable candidates found after Stage 1."
    
    # 2a. Engineer features for the candidate set
    inference_df = stage1_candidates.assign(key=1).merge(patient_info.to_frame().T.assign(key=1), on='key').drop('key', axis=1)
    inference_df['language_match'] = [1 if p_lang in d_langs else 0 for p_lang, d_langs in zip(inference_df['primary_language'], inference_df['languages_spoken'])]
    inference_df['race_match'] = (inference_df['race'] == inference_df['derived_race']).astype(int)
    inference_df['distance_km'] = dist.query_postal_code(inference_df['zip_code'].tolist(), inference_df['practice_zip_code'].tolist())
    inference_df['distance_km'].fillna(inference_df['distance_km'].mean(), inplace=True)

    # 2b. Predict with LTR model
    X_inference = inference_df[features_ltr]
    final_scores = ltr_model.predict(X_inference)
    stage1_candidates['final_score'] = final_scores
    
    # 2c. Rank and return top N
    final_recommendations = stage1_candidates.sort_values(by='final_score', ascending=False)
    
    return final_recommendations.head(num_final_recs)[['provider_id', 'specialty', 'final_score', 'distance_km']]


### Example Usage

In [ ]:
# --- Replace these values with a real patient_id and specialty from your data ---
example_patient_id = 1 # Replace with a valid ID from patient_df
example_specialty = 'Cardiology' # Replace with a valid specialty from provider_df

# Generate recommendations
recommendations = get_two_stage_recommendations(
    patient_id=example_patient_id,
    required_specialty=example_specialty,
    all_providers_df=provider_df,
    all_patients_df=patient_df,
    svd_model=svd,
    ltr_model=ltr_model,
    interaction_matrix=interaction_df_filled,
    patient_map=patient_id_map,
    provider_map=provider_id_map
)

display(recommendations)